# Irrigation Training - v2.19b-TD3 (CORRECTED: anti-collapse exploration)

**This is the corrected re-run of v2.19.** Architecture, env, observation, reward, and the V211 LayerNorm VDN critic are **byte-identical** to v2.19 (deterministic `_TD3SharedActor`, marker 2.19; `runner.py` dispatches it unchanged via marker>=2.185 + no log_std -> `TD3.load`). Only the **training dynamics** change.

## Why v2.19 collapsed (re-derived from the committed eval files)

v2.19 removed the SAC entropy term to unpin the actor from the 6 mm centre so it could reach 0 mm on wet days. It worked too well: the deterministic actor collapsed to ~0 mm in **every** scenario.
- eval mean-reward flat at **~-6** for all 250k steps (never improved; SAC v2.18 sat near 0);
- twin-Q critic **diverged**: q_pred_mean -21 -> -57 -> -105 -> **-145** while realised return was only ~-4;
- trained policy applied <0.5 mm on **69% (dry) / 93% (mod) / 75% (wet)** of days; dry yield **1836** vs SAC **4205** kg/ha, water **85** vs **484** mm.

**Root cause:** the entropy term was load-bearing for **exploration / replay-buffer coverage**, not just the action-pin. v2.19 replaced it with only a weak, fast-decaying noise (0.20->0.05/100k) + target-policy smoothing (which gives **no** state-action coverage). The actor drifted to the 0 mm corner, the buffer filled with drought transitions, twin-min pessimism kept higher-water actions looking bad, and nothing could recover. Aggravated by `learning_starts=1000` and the **5x asymmetric actor LR**.

## The four fixes in v2.19b

| # | knob | v2.19 | **v2.19b** | why |
|---|---|---|---|---|
| 1 | `learning_starts` | 1,000 | **25,000** | warm-start the critic on random data before the actor exploits it |
| 2 | exploration noise | 0.20->0.05 / 100k | **0.40->0.15 / 150k, floor held** | sustained coverage; TD3's ONLY exploration source |
| 3 | `actor_lr_mult` | 5x | **1x** | stop the actor racing to the boundary ahead of the critic |
| 4 | telemetry | (none) | **LowActionCoverage + CollapseGuard** | log coverage; abort early if collapse recurs |

The `CollapseGuard` aborts the run if the rolling low-action fraction exceeds 60% after 30k steps -> a recurrence costs ~35-40k steps, not a full 250k.


In [ ]:
# Clone repo + install deps (SB3 pinned 2.6.0). Same stack as the SAC/TD3 runs.
import subprocess, sys, os
WORK='/kaggle/working'; repo=os.path.join(WORK,'thesis')
if os.path.exists(repo): subprocess.run(['rm','-rf',repo],check=True)
subprocess.run(['git','clone','https://github.com/taratorbati/thesis.git',repo],check=True)
os.chdir(repo); sys.path.insert(0,repo)
subprocess.run(['pip','install','--quiet','stable-baselines3==2.6.0','gymnasium','wandb','pytest'],check=True)
import torch; print(f'PyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY']=UserSecretsClient().get_secret('WANDB_API_KEY'); print('OK WANDB key.')
except Exception as e: print('No WANDB key (%s).'%type(e).__name__)
import subprocess; print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'no GPU')


In [ ]:
# Pre-flight: smoke tests + a TD3 pilot that actually RUNS gradient steps.
# learning_starts=200 < total=1200 so the TD3 update loop executes (critic loss,
# target-policy smoothing, policy_delay, symmetric LR, CollapseGuard wiring) --
# catching runtime bugs before the full run. guard_abort=False so the short
# pilot never false-trips. Then assert the checkpoint is a TD3 actor.
import subprocess, sys
print('Smoke tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_rl_smoke.py','-v','--tb=short']).returncode==0,'SMOKE FAILED'
print('\nFactorized-critic tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_factorized_critic.py','-v','--tb=short']).returncode==0,'CRITIC TESTS FAILED'
print('\nTD3 v2.19b pilot (runs real gradient steps)...')
from src.rl.train_v219b_td3 import train_td3_v219b
m = train_td3_v219b(seed=999, output_dir='/kaggle/working/pilot', wandb_project=None,
                    total_timesteps=1200, learning_starts=200,
                    explore_decay_steps=300, guard_abort=False)
import glob, zipfile, io, torch
ck = glob.glob('/kaggle/working/pilot/td3_v219b_seed999/*final*.zip')
if ck:
    with zipfile.ZipFile(ck[0]) as z:
        sd = torch.load(io.BytesIO(z.open('policy.pth').read()), map_location='cpu', weights_only=False)
    assert 'actor.log_std.weight' not in sd, 'BUG: TD3 actor unexpectedly has log_std!'
    assert 'actor.mu_head.weight' in sd, 'BUG: TD3 actor missing mu_head!'
    assert abs(float(sd['actor.obs_norm_marker'].item()) - 2.19) < 0.01, 'BUG: marker != 2.19'
    print('  Checkpoint sanity: deterministic actor (no log_std), mu_head present, marker=2.19. OK')
print('\nOK pre-flight passed. Proceed.')


In [ ]:
# Full 250k TD3 v2.19d training (= v2.19b architecture + delta-u smoothing r5).
# ~30-55 min A100 / ~2-2.5 h T4. Hyperparameters shown explicitly for the record.
SEED = 0    # CHANGE per session
from src.rl.train_v219b_td3 import train_td3_v219b
model = train_td3_v219b(
    seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    learning_starts=50_000,                                  # hardened early phase (was 25k)
    explore_sigma_start=0.40, explore_sigma_end=0.15,
    explore_decay_steps=150_000,
    actor_lr_mult=1.0,
    target_policy_noise=0.2, target_noise_clip=0.5, policy_delay=2,
    guard_collapse_frac=0.60, guard_warmup_steps=30_000, guard_abort=False,  # log-only, don't kill on transient
    reward_du_alpha=0.005,                                   # v2.19d: delta-u penalty (MPC alpha5)
)
print('Training complete.')

In [ ]:
# Archive results (Kaggle persists /kaggle/working output).
import shutil, os, datetime
src=f'/kaggle/working/thesis/results/rl/td3_v219b_seed{SEED}'
dst=f'/kaggle/working/td3_v219b_seed{SEED}_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
shutil.copytree(src,dst,ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print('Archived to:',dst)
for root,_,files in os.walk(dst):
    for f in files: print(' ',os.path.relpath(os.path.join(root,f),dst))


In [ ]:
# Post-training eval of THIS run's best_model -> per-run subdir (--out-tag),
# never collides with committed dirs / the final eval. --force overwrites stale.
import subprocess, sys, os
model_path=f'/kaggle/working/thesis/results/rl/td3_v219b_seed{SEED}/best_model/best_model.zip'
final_path=f'/kaggle/working/thesis/results/rl/td3_v219b_seed{SEED}/td3_v219b_seed{SEED}_final.zip'
BEST_TAG=f'eval_best_seed{SEED}'; FINAL_TAG=f'eval_final_seed{SEED}'
print('Evaluating BEST (perfect)...')
r=subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','perfect',
    '--force','--out-tag',BEST_TAG],capture_output=True,text=True)
print(r.stdout[-1500:])
if r.returncode!=0: print('STDERR:',r.stderr[-2500:])
assert r.returncode==0,'PERFECT EVAL FAILED'
print('\nEvaluating BEST (noisy, robustness)...')
subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','noisy',
    '--noise-seed','42','--force','--out-tag',BEST_TAG])
if os.path.exists(final_path):
    print('\nEvaluating FINAL (250k, perfect)...')
    subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
        '--model',final_path,'--scenario','all','--budget','all','--forecast','perfect',
        '--force','--out-tag',FINAL_TAG])
print(f'\nBest-model eval -> results/runs/{BEST_TAG}/')

In [ ]:
# PRIMARY DIAGNOSTIC + NO-COLLAPSE GATE. Reads THIS run's best_model eval
# from the explicit per-run subdir (no mtime glob -> no misattribution).
import pandas as pd, numpy as np, json, glob, os
OUT=f'/kaggle/working/thesis/results/runs/eval_best_seed{SEED}'
print('Eval dir:',OUT)
assert os.path.isdir(OUT),f'{OUT} missing -- run the eval cell first.'

def agg(d, scen):
    ys,ws,wls,x1s,umeans=[],[],[],[],[]
    for b in ['100pct','85pct','70pct']:
        pj=glob.glob(os.path.join(d,f'sac_perfect_det_{scen}_rice_{b}_seed*.json'))
        pq=glob.glob(os.path.join(d,f'sac_perfect_det_{scen}_rice_{b}_seed*.parquet'))
        if not pj or not pq: continue
        m=json.load(open(pj[0]))['final_metrics']; df=pd.read_parquet(pq[0])
        ys.append(m['yield_kg_ha']); ws.append(m.get('water_used_mm'))
        wls.append(m.get('waterlog_days_per_agent')); x1s.append(float(df['x1'].median()))
        umeans.append(float(df['u'].mean()))
    f=lambda a:float(np.mean([v for v in a if v is not None])) if a else float('nan')
    return f(ys),f(ws),f(wls),f(x1s),f(umeans)

y,w,wl,x1,_=agg(OUT,'wet'); dyld,_,_,_,du=agg(OUT,'dry')
print('='*58)
print(f'{"metric":<16}{"v2.19d":>9}{"v2.18":>8}{"MPC":>7}{"target":>9}')
print(f'{"wet x1 median":<16}{x1:9.1f}{136:8.0f}{130:7.0f}{"<134":>9}')
print(f'{"wet waterlog":<16}{wl:9.1f}{37:8.0f}{18:7.0f}{"<32":>9}')
print(f'{"wet water mm":<16}{w:9.0f}{336:8.0f}{309:7.0f}{"<330":>9}')
print(f'{"wet yield":<16}{y:9.0f}{3687:8.0f}{3752:7.0f}{"(info)":>9}')   # kg/ha (NOT x10)
print('-'*58)
print('NO-COLLAPSE GATE (v2.19 collapsed here):')
print(f'  dry u_mean = {du:.2f} mm/day   (v2.19 collapse: 0.92; SAC v2.18: 5.20; want >~4)')
print(f'  dry yield  = {dyld:.0f} kg/ha   (v2.19 collapse: 1836; SAC v2.18: 4205)')
print(f'  {"PASS - no collapse" if du>4.0 else "FAIL - collapsing!"}')
print('PRIMARY:')
print(f'  wet x1 < 134      : {"PASS" if x1<134 else "FAIL"} ({x1:.1f})')
print(f'  wet waterlog < 32 : {"PASS" if wl<32 else "FAIL"} ({wl:.1f})')

In [ ]:
# STABILITY + COLLAPSE-GUARD DIAGNOSTIC.
import os, glob, numpy as np, pandas as pd
run_dir=f'/kaggle/working/thesis/results/rl/td3_v219b_seed{SEED}'
br=os.path.join(run_dir,'bias_ratio_log.csv')
if os.path.exists(br):
    b=pd.read_csv(br); print('--- bias_ratio_log ---'); print(b.to_string(index=False))
    neg=(b['q_pred_mean']<0).any()
    mx=b['q_inflation_pct'].abs().max()
    q_final=float(b['q_pred_mean'].iloc[-1])
    # TD3 has NO entropy term -> q_structural=0 -> q_inflation_pct is NaN by
    # construction, so the old gate (mx<80) is undefined and ALWAYS fails on NaN.
    # For entropy-free TD3 judge calibration by whether q_pred RECOVERS instead.
    if np.isfinite(mx):
        calibrated=(not neg) and (mx<80)
        print(f'\n  q_pred ever negative: {neg};  final q_pred: {q_final:+.2f}')
        print(f'  max |q_inflation_pct|: {mx:.1f}%  (target < 80)')
        verdict='CALIBRATED critic' if calibrated else 'still miscalibrated -- investigate'
    else:
        calibrated=(q_final >= -1.0)
        print(f'\n  q_inflation_pct: N/A (entropy-free TD3, q_structural=0)')
        print(f'  q_pred trajectory: min={b["q_pred_mean"].min():+.1f}  final={q_final:+.2f}')
        verdict=('q_pred RECOVERED -> calibrated for TD3' if calibrated
                 else f'q_pred still low ({q_final:+.2f}) -- critic over-pessimistic, investigate')
    print(f'  VERDICT: {verdict}')
else:
    print('No bias_ratio_log.csv at', br)

cg=os.path.join(run_dir,'collapse_guard_log.csv')
if os.path.exists(cg):
    g=pd.read_csv(cg)
    tripped=int(g['collapsed'].max()) if len(g) else 0
    print(f'\n--- collapse_guard ---  rows={len(g)}  final rolling low-frac={g["frac_low_rolling"].iloc[-1]:.0%}')
    print(f'  guard tripped: {"YES (collapsed)" if tripped else "NO (healthy)"}')
cov=os.path.join(run_dir,'low_action_coverage_log.csv')
if os.path.exists(cov):
    c=pd.read_csv(cov)
    print(f'--- low_action_coverage ---  mean frac_low (last 50)={c["frac_low_action"].tail(50).mean():.0%}')

try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    runs=glob.glob(os.path.join(run_dir,'tensorboard','*'))
    if runs:
        ea=EventAccumulator(runs[0]); ea.Reload()
        if 'train/critic_loss' in ea.Tags()['scalars']:
            cl=ea.Scalars('train/critic_loss'); mxl=max(e.value for e in cl)
            print(f'\n  max critic_loss = {mxl:.2f}  (STABLE if < 100; v2.7 cascade hit 6.9e12)')
except Exception as e:
    print('tensorboard read skipped:', e)

In [ ]:
# Resume from checkpoint.
# SEED=0; STEP=100_000
# CKPT=f'/kaggle/input/<your-dataset>/td3_v219b_seed{SEED}/checkpoints/td3_v219b_seed{SEED}_{STEP}_steps.zip'
# from src.rl.train_v219b_td3 import AsymmetricLRTD3
# from src.rl.networks_td3 import TD3VDNPolicy
# model=AsymmetricLRTD3.load(CKPT, custom_objects={'policy_class':TD3VDNPolicy})
# # model.learn(total_timesteps=..., reset_num_timesteps=False)
# # NOTE: action_noise not restored; recreate NormalActionNoise + ExplorationNoiseDecayCallback
# #       (sigma_start=0.40, sigma_end=0.15, decay_steps=150_000) + CollapseGuardCallback.
